# Architecture Evaluation — All 8 ResNet18 Variants

Trains all 8 attention-augmented ResNet18 architectures under 5-fold stratified CV using the optimal hyperparameters from the grid search and the winning head type from the ablation study. Weights are saved per architecture per fold for use in downstream NCA+kNN experiments (SRQ2).

**Architectures (8 total):**
- `base` — plain ResNet18 baseline
- `cbam_end` — single CBAM after avgpool
- `cbam_block_pre` — CBAM inside each block, pre-shortcut
- `cbam_block_post` — CBAM inside each block, post-shortcut
- `se_end` — single SE block after avgpool
- `se_block_pre` — SE inside each block, pre-shortcut
- `cbam_isolated_end` — CBAMIsolated (SE channel + CBAM spatial) after avgpool
- `cbam_isolated_block_pre` — CBAMIsolated inside each block, pre-shortcut

**Protocol:**
- Head type and hyperparameters loaded automatically from prior experiment outputs
- 5-fold stratified CV; 8 architectures × 5 folds = 40 runs
- Weights saved to `arch-eval-results/weights/<architecture>/fold_<n>.pt` after each fold
- Results saved fold-by-fold to `arch-eval-results/arch_eval_results.csv` — safe to interrupt and resume
- **Selection criterion:** highest mean val F1. Ties: highest mean val acc, then lowest mean val loss.

## 1 · Paths & Imports

In [7]:
import sys
from pathlib import Path

ABSOLUTE_PATH = Path().resolve()
PROJECT_ROOT  = ABSOLUTE_PATH.parents[2]
DATA_DIR      = PROJECT_ROOT / "data" / "raw"
RESULTS_DIR   = ABSOLUTE_PATH / "arch-eval-results"
WEIGHTS_DIR   = RESULTS_DIR / "weights"
CURVES_DIR   = RESULTS_DIR / "training-curves"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
WEIGHTS_DIR.mkdir(parents=True, exist_ok=True)
CURVES_DIR.mkdir(parents=True, exist_ok=True)

sys.path.append(str(PROJECT_ROOT))

print(f"Project root : {PROJECT_ROOT}")
print(f"Data dir     : {DATA_DIR}")
print(f"Results dir  : {RESULTS_DIR}")
print(f"Weights dir  : {WEIGHTS_DIR}")
print(f"Curves dir   : {CURVES_DIR}")

Project root : C:\Users\markm\Workspace\ms-machine-learning-diagnosis
Data dir     : C:\Users\markm\Workspace\ms-machine-learning-diagnosis\data\raw
Results dir  : C:\Users\markm\Workspace\ms-machine-learning-diagnosis\src\notebooks\CNNS\arch-eval-results
Weights dir  : C:\Users\markm\Workspace\ms-machine-learning-diagnosis\src\notebooks\CNNS\arch-eval-results\weights
Curves dir   : C:\Users\markm\Workspace\ms-machine-learning-diagnosis\src\notebooks\CNNS\arch-eval-results\training-curves


In [8]:
import csv
import traceback
from datetime import datetime

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker

from sklearn.metrics import f1_score

import src.scripts.data    as data
import src.scripts.models  as models
import src.scripts.trainer as trainer
import src.scripts.utils   as utils

utils.set_seed(42)

Random seed set to 42 for Python, NumPy, and PyTorch


## 2 · Data — Outer Split (Identical to All Other Experiments)

Fixed seed 42 outer split. Architecture evaluation operates entirely within `X_trainval`; the held-out test set remains untouched until final evaluation.

In [9]:
path, categories = data.get_dataset(str(DATA_DIR))
classes = data.get_classes(path, categories, visualise=False)

image_paths, labels = data.get_paths_and_labels(path, classes)
train_transform, test_transform = data.get_transforms()

X_trainval, y_trainval, X_test, y_test = data.get_trainval_test_split(
    image_paths, labels,
    test_split=0.20,
    SEED=42
)

print(f"\nArchitecture evaluation operates on {len(X_trainval)} train+val samples.")
print("Held-out test set is NOT used at any point in this notebook.")

get_dataset()>>> Dataset already exists in C:\Users\markm\Workspace\ms-machine-learning-diagnosis\data\raw
get_dataset()>>> Available categories: ['Control Axial_crop', 'Control Saggital_crop', 'MS Axial_crop', 'MS Saggital_crop']
get_paths_and_labels()>>> Total images: 1652
get_trainval_test_split()>>> Train+Val pool : 1321 (80.0%)
get_trainval_test_split()>>> Held-out test  : 331 (20.0%)
get_trainval_test_split()>>> TrainVal class ratio — MS: 520  Non-MS: 801
get_trainval_test_split()>>> Test     class ratio — MS: 130  Non-MS: 201

Architecture evaluation operates on 1321 train+val samples.
Held-out test set is NOT used at any point in this notebook.


## 3 · Configuration

Hyperparameters are loaded from `grid-search-results/optimal_config.csv` (grid search output). The winning head type is loaded from `head-ablation-results/optimal_head.csv` (ablation output). Both files must exist before running this notebook.

In [10]:
# ── Load optimal hyperparameters from grid search ────────────────────────
OPTIMAL_CONFIG_FILE = ABSOLUTE_PATH / "grid-search-results" / "optimal_config.csv"
if not OPTIMAL_CONFIG_FILE.exists():
    raise FileNotFoundError(
        f"optimal_config.csv not found at {OPTIMAL_CONFIG_FILE}\n"
        "Run the grid search notebook (Section 5) first."
    )

optimal = pd.read_csv(OPTIMAL_CONFIG_FILE).iloc[0]
LR_PHASE1         = float(optimal["lr_phase1"])
WD_PHASE1         = float(optimal["wd_phase1"])
BEST_LR_PHASE2    = float(optimal["lr_phase2"])
BEST_WEIGHT_DECAY = float(optimal["weight_decay"])

print(f"Loaded from : {OPTIMAL_CONFIG_FILE.name}")
print(f"  lr_phase1    = {LR_PHASE1:.0e}")
print(f"  wd_phase1    = {WD_PHASE1}")
print(f"  lr_phase2    = {BEST_LR_PHASE2:.0e}")
print(f"  weight_decay = {BEST_WEIGHT_DECAY:.0e}")

# ── Load winning head from ablation optimal_head.csv ─────────────────────────
OPTIMAL_HEAD_FILE = ABSOLUTE_PATH / "head-ablation-results" / "optimal_head.csv"
if not OPTIMAL_HEAD_FILE.exists():
    raise FileNotFoundError(
        f"optimal_head.csv not found at {OPTIMAL_HEAD_FILE}\n"
        "Run the head ablation notebook (Section 5 + save cell) first."
    )

optimal_head = pd.read_csv(OPTIMAL_HEAD_FILE).iloc[0]
WINNING_HEAD = optimal_head["head"]

print(f"Loaded from : {OPTIMAL_HEAD_FILE.name}")
print(f"  winning head = {WINNING_HEAD}")
print(f"  mean_val_f1  = {float(optimal_head['mean_val_f1']):.4f}")
print(f"  mean_val_acc = {float(optimal_head['mean_val_acc']):.4f}")

Loaded from : optimal_config.csv
  lr_phase1    = 1e-03
  wd_phase1    = 0.0
  lr_phase2    = 1e-04
  weight_decay = 1e-04
Loaded from : optimal_head.csv
  winning head = linear
  mean_val_f1  = 0.9174
  mean_val_acc = 0.9372


In [11]:
# ── Fixed training parameters ─────────────────────────────────────────────
N_SPLITS    = 5
BATCH_SIZE  = 32
P1_EPOCHS   = 20
P2_EPOCHS   = 15
P1_PATIENCE = 4
P2_PATIENCE = 3
SEED        = 42

# ── Architecture list — order determines run sequence ─────────────────────
ARCHITECTURES = [
    "base",
    "cbam_end",
    "cbam_block_pre",
    "cbam_block_post",
    "se_end",
    "se_block_pre",
    "cbam_isolated_end",
    "cbam_isolated_block_pre",
]

RESULTS_FILE = RESULTS_DIR / "arch_eval_results.csv"
CSV_FIELDNAMES = [
    "architecture", "head", "lr_phase1", "wd_phase1", "lr_phase2", "weight_decay",
    "fold", "val_acc", "val_loss", "val_f1",
    "p1_epochs_run", "p2_epochs_run",
    "weights_path", "timestamp", "error"
]

total_runs = len(ARCHITECTURES) * N_SPLITS
print(f"{len(ARCHITECTURES)} architectures × {N_SPLITS} folds = {total_runs} runs")
print(f"Head      : {WINNING_HEAD}")
print(f"lr_phase2 : {BEST_LR_PHASE2:.0e}  weight_decay : {BEST_WEIGHT_DECAY:.0e}")
print(f"Results   → {RESULTS_FILE}")
print(f"Weights   → {WEIGHTS_DIR}/<architecture>/fold_<n>.pt")

8 architectures × 5 folds = 40 runs
Head      : linear
lr_phase2 : 1e-04  weight_decay : 1e-04
Results   → C:\Users\markm\Workspace\ms-machine-learning-diagnosis\src\notebooks\CNNS\arch-eval-results\arch_eval_results.csv
Weights   → C:\Users\markm\Workspace\ms-machine-learning-diagnosis\src\notebooks\CNNS\arch-eval-results\weights/<architecture>/fold_<n>.pt


## 4 · Helpers

Resume logic keys on `(architecture, fold)` — re-running the training cell skips completed pairs. A crash or kernel restart loses at most one fold.

In [12]:
def load_completed_runs(results_file):
    """Return set of (architecture, fold) tuples already written without error."""
    completed = set()
    if results_file.exists():
        with open(results_file, newline="") as f:
            reader = csv.DictReader(f)
            for row in reader:
                if row["error"] == "":
                    completed.add((row["architecture"], int(row["fold"])))
    return completed


def append_result(results_file, fieldnames, row_dict):
    """Append one result row to CSV, writing header if the file is new."""
    write_header = not results_file.exists()
    with open(results_file, "a", newline="") as f:
        writer = csv.DictWriter(f, fieldnames=fieldnames)
        if write_header:
            writer.writeheader()
        writer.writerow(row_dict)


def weights_path_for(arch, fold_idx):
    """Return the .pt save path for a given architecture and fold."""
    arch_dir = WEIGHTS_DIR / arch
    arch_dir.mkdir(parents=True, exist_ok=True)
    return arch_dir / f"fold_{fold_idx}.pt"

## 5 · Training Loop

Outer loop: architectures. Inner loop: folds. After each fold: weights saved to `weights/<arch>/fold_<n>.pt`, training curves saved to `training-curves/<arch>/phase1_fold<n>_curves.png` and `phase2_fold<n>_curves.png`. `show=False` keeps the notebook output clean during long runs. Expected duration: ~20 min/run × 40 runs ≈ 13 hours on CPU.

In [13]:
completed_runs = load_completed_runs(RESULTS_FILE)
run_number     = len(completed_runs)

for arch in ARCHITECTURES:
    for fold_idx in range(N_SPLITS):

        run_key = (arch, fold_idx)

        if run_key in completed_runs:
            print(f"SKIP  arch={arch}  fold={fold_idx+1}/{N_SPLITS}")
            continue

        utils.set_seed(SEED)
        run_number += 1
        print(f"\n{'='*65}")
        print(f"  Run {run_number}/{total_runs}  |  arch={arch}  fold={fold_idx+1}/{N_SPLITS}")
        print(f"{'='*65}")

        try:
            train_loader, val_loader = data.get_fold_loaders(
                X_trainval, y_trainval,
                fold_idx=fold_idx,
                train_transform=train_transform,
                test_transform=test_transform,
                n_splits=N_SPLITS,
                batch_size=BATCH_SIZE,
                SEED=SEED
            )

            model = models.get_model(architecture=arch, head=WINNING_HEAD)

            run_configs = {
                "phase1": {
                    "num_epochs"  : P1_EPOCHS,
                    "lr"          : LR_PHASE1,
                    "parameters"  : "head_and_attention",
                    "optimiser"   : optim.AdamW,
                    "criterion"   : nn.BCEWithLogitsLoss(),
                    "weight_decay": WD_PHASE1,
                },
                "phase2": {
                    "num_epochs"  : P2_EPOCHS,
                    "lr"          : BEST_LR_PHASE2,
                    "parameters"  : "all",
                    "optimiser"   : optim.AdamW,
                    "criterion"   : nn.BCEWithLogitsLoss(),
                    "weight_decay": BEST_WEIGHT_DECAY,
                },
            }

            # Phase 1: head + attention warm-up
            losses_p1, accs_p1, val_losses_p1, val_accs_p1 = trainer.train_model(
                model, train_loader, val_loader,
                config_name="phase1",
                train_configs=run_configs,
                early_stopping_patience=P1_PATIENCE
            )
            p1_epochs_run = len(val_accs_p1)

            utils.plot(
                losses_p1, accs_p1,
                config_name=f"phase1_fold{fold_idx}",
                val_losses=val_losses_p1, val_accuracies=val_accs_p1,
                model_name=arch,
                save_dir=str(CURVES_DIR),
                show=False
            )

            # Phase 2: full fine-tune
            losses_p2, accs_p2, val_losses_p2, val_accs_p2 = trainer.train_model(
                model, train_loader, val_loader,
                config_name="phase2",
                train_configs=run_configs,
                early_stopping_patience=P2_PATIENCE
            )
            p2_epochs_run = len(val_accs_p2)

            utils.plot(
                losses_p2, accs_p2,
                config_name=f"phase2_fold{fold_idx}",
                val_losses=val_losses_p2, val_accuracies=val_accs_p2,
                model_name=arch,
                save_dir=str(CURVES_DIR),
                show=False
            )

            # F1 inference pass
            model.eval()
            all_preds, all_labels_list = [], []
            with torch.no_grad():
                for imgs, lbls in val_loader:
                    imgs   = imgs.to(next(model.parameters()).device)
                    logits = model(imgs).squeeze(1)
                    preds  = (torch.sigmoid(logits) >= 0.5).long().cpu().tolist()
                    all_preds.extend(preds)
                    all_labels_list.extend(lbls.tolist())
            val_f1 = f1_score(all_labels_list, all_preds, average="binary", zero_division=0)

            # Save weights
            wpath = weights_path_for(arch, fold_idx)
            utils.save_weights(model, wpath)

            result_row = {
                "architecture" : arch,
                "head"         : WINNING_HEAD,
                "lr_phase1"    : LR_PHASE1,
                "wd_phase1"    : WD_PHASE1,
                "lr_phase2"    : BEST_LR_PHASE2,
                "weight_decay" : BEST_WEIGHT_DECAY,
                "fold"         : fold_idx,
                "val_acc"      : round(float(val_accs_p2[-1]),  6),
                "val_loss"     : round(float(val_losses_p2[-1]), 6),
                "val_f1"       : round(float(val_f1), 6),
                "p1_epochs_run": p1_epochs_run,
                "p2_epochs_run": p2_epochs_run,
                "weights_path" : str(wpath),
                "timestamp"    : datetime.now().isoformat(timespec="seconds"),
                "error"        : "",
            }
            append_result(RESULTS_FILE, CSV_FIELDNAMES, result_row)
            completed_runs.add(run_key)

            print(f"  val_acc={val_accs_p2[-1]:.4f}  "
                  f"val_f1={val_f1:.4f}  "
                  f"val_loss={val_losses_p2[-1]:.4f}  "
                  f"(p1={p1_epochs_run}ep  p2={p2_epochs_run}ep)")
            print(f"  Weights → {wpath}")
            print(f"  Curves  → {CURVES_DIR}/{arch}/")

        except Exception as e:
            error_msg = f"{type(e).__name__}: {str(e)}"
            print(f"ERROR -- {error_msg}")
            traceback.print_exc()
            error_row = {
                "architecture" : arch,
                "head"         : WINNING_HEAD,
                "lr_phase1"    : LR_PHASE1,    "wd_phase1"    : WD_PHASE1,
                "lr_phase2"    : BEST_LR_PHASE2, "weight_decay": BEST_WEIGHT_DECAY,
                "fold"         : fold_idx,
                "val_acc"      : "", "val_loss": "", "val_f1": "",
                "p1_epochs_run": "", "p2_epochs_run": "",
                "weights_path" : "",
                "timestamp"    : datetime.now().isoformat(timespec="seconds"),
                "error"        : error_msg,
            }
            append_result(RESULTS_FILE, CSV_FIELDNAMES, error_row)

print(f"\n{'='*65}")
print("ARCHITECTURE EVALUATION COMPLETE")
print(f"Results  → {RESULTS_FILE}")
print(f"Weights  → {WEIGHTS_DIR}")
print(f"Curves   → {CURVES_DIR}")

Random seed set to 42 for Python, NumPy, and PyTorch

  Run 1/40  |  arch=base  fold=1/5
get_fold_loaders()>>> Fold 1/5 — Train: 1056,  Val: 265
get_model()>>> architecture='base'  head='linear'


KeyboardInterrupt: 

## 6 · Results Analysis

Run after training completes (or at any point during it for partial results).

In [ ]:
df_raw = pd.read_csv(RESULTS_FILE)
df_raw["error"] = df_raw["error"].fillna("")

df_ok   = df_raw[df_raw["error"] == ""].copy()
df_fail = df_raw[df_raw["error"] != ""].copy()

for col in ["val_f1", "val_acc", "val_loss"]:
    df_ok[col] = df_ok[col].astype(float)

print(f"Successful runs : {len(df_ok)} / {total_runs}")
print(f"Failed runs     : {len(df_fail)}")
if len(df_fail) > 0:
    print(df_fail[["architecture", "fold", "error"]].to_string(index=False))

In [ ]:
# Summary table — mean ± std across 5 folds per architecture
summary = (
    df_ok
    .groupby("architecture")
    .agg(
        mean_val_f1   = ("val_f1",        "mean"),
        std_val_f1    = ("val_f1",        "std"),
        mean_val_acc  = ("val_acc",       "mean"),
        std_val_acc   = ("val_acc",       "std"),
        mean_val_loss = ("val_loss",      "mean"),
        std_val_loss  = ("val_loss",      "std"),
        mean_p2_epochs= ("p2_epochs_run", "mean"),
        n_folds       = ("val_f1",        "count"),
    )
    .reset_index()
    .sort_values("mean_val_f1", ascending=False)
)

# Preserve architecture order for display even after sort
arch_order = {a: i for i, a in enumerate(ARCHITECTURES)}

summary["val_f1_fmt"]  = summary.apply(lambda r: f"{r.mean_val_f1:.4f} ± {r.std_val_f1:.4f}", axis=1)
summary["val_acc_fmt"] = summary.apply(lambda r: f"{r.mean_val_acc:.4f} ± {r.std_val_acc:.4f}", axis=1)
summary["val_loss_fmt"]= summary.apply(lambda r: f"{r.mean_val_loss:.4f} ± {r.std_val_loss:.4f}", axis=1)

display_cols = ["architecture", "val_f1_fmt", "val_acc_fmt", "val_loss_fmt",
                "mean_p2_epochs", "n_folds"]

print("\nArchitecture evaluation summary (sorted by mean val F1):\n")
summary[display_cols]

In [ ]:
# Winning architecture
best = summary.sort_values(
    ["mean_val_f1", "mean_val_acc", "mean_val_loss"],
    ascending=[False, False, True]
).iloc[0]

baseline = summary[summary["architecture"] == "base"].iloc[0]

print("\n" + "="*60)
print("  WINNING ARCHITECTURE")
print("="*60)
print(f"  Architecture  : {best['architecture']}")
print(f"  Mean val F1   : {best.mean_val_f1:.4f} ± {best.std_val_f1:.4f}")
print(f"  Mean val acc  : {best.mean_val_acc:.4f} ± {best.std_val_acc:.4f}")
print(f"  Mean val loss : {best.mean_val_loss:.4f} ± {best.std_val_loss:.4f}")
print("="*60)
print(f"\n  ΔF1 vs baseline  : {best.mean_val_f1 - baseline.mean_val_f1:+.4f}")
print(f"  ΔAcc vs baseline : {best.mean_val_acc  - baseline.mean_val_acc:+.4f}")

WINNING_ARCH = best["architecture"]
print(f"\nWINNING_ARCH = '{WINNING_ARCH}'")
print("\nCarry WINNING_ARCH into the NCA+kNN notebook (SRQ2).")

In [ ]:
# Per-fold breakdown for each architecture
print("Per-fold results:\n")
for arch in ARCHITECTURES:
    rows = df_ok[df_ok["architecture"] == arch].sort_values("fold")
    if rows.empty:
        print(f"  {arch}: no completed folds yet")
        continue
    print(f"  {arch}")
    for _, r in rows.iterrows():
        print(f"    Fold {int(r.fold)+1}: "
              f"val_f1={r.val_f1:.4f}  "
              f"val_acc={r.val_acc:.4f}  "
              f"val_loss={r.val_loss:.4f}  "
              f"(p1={int(r.p1_epochs_run)}ep  p2={int(r.p2_epochs_run)}ep)")
    print(f"    → mean F1={rows.val_f1.mean():.4f}  std={rows.val_f1.std():.4f}\n")

In [ ]:
# Save summary
summary_path = RESULTS_DIR / "arch_eval_summary.csv"
summary[display_cols].to_csv(summary_path, index=False)
print(f"Summary saved to: {summary_path}")

## 7 · Visualisation

Horizontal bar chart ranked by mean val F1, with ±1 std error bars and individual fold values overlaid. Baseline (`base`) highlighted for reference.

In [ ]:
if len(df_ok) == 0:
    print("No completed runs yet — run Section 5 first.")
else:
    # Sort by mean F1 descending for the chart
    plot_df = summary.sort_values("mean_val_f1", ascending=True)  # ascending=True → highest at top

    fig, ax = plt.subplots(figsize=(11, 6))

    colours = ["#DD8452" if a == "base" else "#4C72B0" for a in plot_df["architecture"]]
    y_pos   = np.arange(len(plot_df))

    ax.barh(y_pos, plot_df["mean_val_f1"], xerr=plot_df["std_val_f1"],
            color=colours, alpha=0.82, height=0.55,
            capsize=4, error_kw=dict(elinewidth=1.2, ecolor="#555", capthick=1.2),
            edgecolor="white", linewidth=0.8)

    # Overlay individual fold values
    for i, arch in enumerate(plot_df["architecture"]):
        fold_f1 = df_ok.loc[df_ok["architecture"] == arch, "val_f1"].values
        jitter  = np.random.default_rng(0).uniform(-0.15, 0.15, size=len(fold_f1))
        ax.scatter(fold_f1, i + jitter,
                   color="#DD8452" if arch == "base" else "#4C72B0",
                   edgecolors="white", linewidths=0.6, s=22, zorder=3, alpha=0.9)

    ax.set_yticks(y_pos)
    ax.set_yticklabels(plot_df["architecture"], fontsize=10)
    ax.set_xlabel("Mean Val F1", fontsize=11)
    ax.set_title(
        f"Architecture evaluation — mean val F1 (5-fold CV)\n"
        f"head={WINNING_HEAD}  lr_p2={BEST_LR_PHASE2:.0e}  wd={BEST_WEIGHT_DECAY:.0e}",
        fontsize=11
    )
    ax.xaxis.set_major_formatter(mticker.FormatStrFormatter("%.3f"))
    ax.spines[["top", "right"]].set_visible(False)

    # Zoom x-axis to meaningful range
    all_f1 = df_ok["val_f1"].values
    pad = (all_f1.max() - all_f1.min()) * 0.3 or 0.02
    ax.set_xlim(all_f1.min() - pad, all_f1.max() + pad)

    # Baseline reference line
    if "base" in summary["architecture"].values:
        base_f1 = summary.loc[summary["architecture"] == "base", "mean_val_f1"].values[0]
        ax.axvline(base_f1, color="#DD8452", linestyle="--", linewidth=1.2,
                   alpha=0.7, label=f"Baseline ({base_f1:.4f})")
        ax.legend(fontsize=9)

    plt.tight_layout()
    plt.savefig(RESULTS_DIR / "arch_eval_plot.png", dpi=150, bbox_inches="tight")
    plt.show()
    print("Plot saved to arch-eval-results/arch_eval_plot.png")

## 8 · Weights Index

Verify all expected weight files are present. This index is used by the NCA+kNN notebook to locate trained backbones.

In [ ]:
print("Saved weights:\n")
missing = []
for arch in ARCHITECTURES:
    for fold_idx in range(N_SPLITS):
        wpath = weights_path_for(arch, fold_idx)
        status = "✓" if wpath.exists() else "✗ MISSING"
        print(f"  {status}  {wpath.relative_to(RESULTS_DIR)}")
        if not wpath.exists():
            missing.append(wpath)

print()
if missing:
    print(f"WARNING: {len(missing)} weight file(s) missing — re-run Section 5 to complete.")
else:
    print(f"All {len(ARCHITECTURES) * N_SPLITS} weight files present.")

## 9 · Next Steps

1. **Record `WINNING_ARCH`** from Section 6 — carry it into the NCA+kNN notebook.
2. **SRQ2 — NCA+kNN:** Load the saved backbone weights for the best architecture (and runner-up) using `utils.load_weights()`, extract 512-dim embeddings via `trainer.get_features()`, then fit the NCA+kNN pipeline. The `weights_path` column in `arch_eval_results.csv` gives the exact path for each fold.
3. **Final evaluation:** After all model selection decisions are made, load the best architecture weights and call `evaluator.evaluate_model()` with the held-out test loader from `data.get_test_loader()` — exactly once.